# Eksperimen Deep Learning (Versi 2-Kelas): Dynamic NN vs Static Transformer vs Early-Exit Baselines

**PENTING:** Notebook ini memakai skema **2-kelas (positive/negative)** yang sudah diselaraskan di ketiga bahasa -- versi ini menggantikan versi 3-kelas sebelumnya.

**Cara pakai:** Runtime > Change runtime type > pilih **GPU (T4)**, lalu jalankan semua sel dari atas ke bawah.

Anda akan diminta upload **7 file CSV 2-kelas** di Sel 2:
`indotrain_2class.csv`, `indovalid_2class.csv`, `indotest_2class.csv`, `inggristrain_2class.csv`, `inggrisvalid_2class.csv`, `inggristest_2class.csv`, `melayu_2class.csv`

Di akhir, notebook menghasilkan `results_id.json`, `results_en.json`, `results_ms.json` yang HARUS Anda download dan kirim balik apa adanya (jangan diedit manual) untuk disusun jadi draf jurnal.

In [ ]:
# Sel 1 -- Instalasi
!pip install -q transformers datasets scikit-learn scipy torch accelerate

In [ ]:
# Sel 2 -- Upload 7 file CSV 2-KELAS Anda
from google.colab import files
print('Silakan upload TUJUH file berikut (nama harus persis sama):')
print('indotrain_2class.csv, indovalid_2class.csv, indotest_2class.csv,')
print('inggristrain_2class.csv, inggrisvalid_2class.csv, inggristest_2class.csv,')
print('melayu_2class.csv')
uploaded = files.upload()

In [ ]:
# Sel 3 -- Import & cek GPU
import torch, time, json, numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('PERINGATAN: GPU tidak aktif. Runtime > Change runtime type > GPU (T4), lalu Run all ulang.')

In [ ]:
# Sel 4 -- Load & verifikasi data 2-KELAS (memakai file yang baru diupload)
idtr = pd.concat([pd.read_csv('indotrain_2class.csv'), pd.read_csv('indovalid_2class.csv')])
idte = pd.read_csv('indotest_2class.csv')
entr = pd.concat([pd.read_csv('inggristrain_2class.csv'), pd.read_csv('inggrisvalid_2class.csv')])
ente = pd.read_csv('inggristest_2class.csv')
mal = pd.read_csv('melayu_2class.csv')
mtr, mte = train_test_split(mal, test_size=0.2, stratify=mal['label'], random_state=42)

DATASETS = {
    'id': (idtr['text'].tolist(), idtr['label'].tolist(), idte['text'].tolist(), idte['label'].tolist()),
    'en': (entr['text'].tolist(), entr['label'].tolist(), ente['text'].tolist(), ente['label'].tolist()),
    'ms': (mtr['text'].tolist(), mtr['label'].tolist(), mte['text'].tolist(), mte['label'].tolist()),
}
for lang, (trX, trY, teX, teY) in DATASETS.items():
    classes = sorted(set(trY))
    assert len(classes) == 2, f'ERROR: {lang} punya {len(classes)} kelas, seharusnya 2! Cek file upload Anda.'
    print(lang, '| train:', len(trX), '| test:', len(teX), '| classes:', classes)
print('\nVerifikasi OK: semua bahasa 2 kelas (biner).')

## Model 1: Static mBERT (baseline)

In [ ]:
# Sel 5 -- Fine-tuning mBERT statis + pengukuran latensi
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

MODEL_NAME = 'bert-base-multilingual-cased'

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(list(labels))
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

def fine_tune_mbert(train_texts, train_labels, test_texts, test_labels, num_labels, epochs=5, seed=42):
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels).to(DEVICE)
    train_ds = TextDataset(train_texts, train_labels, tok)
    test_ds = TextDataset(test_texts, test_labels, tok)
    args = TrainingArguments(output_dir='./out', num_train_epochs=epochs, per_device_train_batch_size=16,
                              learning_rate=2e-5, warmup_ratio=0.1, logging_steps=50, save_strategy='no',
                              eval_strategy='no', seed=seed, report_to=[])
    trainer = Trainer(model=model, args=args, train_dataset=train_ds)
    trainer.train()
    pred_out = trainer.predict(test_ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    acc = accuracy_score(test_labels, preds); f1 = f1_score(test_labels, preds, average='macro')

    model.eval()
    enc = tok(list(test_texts[:50]), truncation=True, padding='max_length', max_length=128, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        for i in range(50):
            idx = i % 50
            _ = model(input_ids=enc['input_ids'][idx:idx+1], attention_mask=enc['attention_mask'][idx:idx+1])
        lat = []
        for i in range(300):
            idx = i % 50
            if DEVICE == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(input_ids=enc['input_ids'][idx:idx+1], attention_mask=enc['attention_mask'][idx:idx+1])
            if DEVICE == 'cuda': torch.cuda.synchronize()
            lat.append((time.perf_counter()-t0)*1000)
    return {'model':'mBERT-static','accuracy':float(acc),'f1_macro':float(f1),
            'latency_ms_mean':float(np.mean(lat)),'latency_ms_std':float(np.std(lat)),'raw_latencies':lat}

## Model 2-4: Early-Exit Baselines (DeeBERT, FastBERT, PABEE)

In [ ]:
# Sel 6 -- Backbone bersama dengan classifier di tiap layer
import torch.nn as nn
from transformers import AutoModel

class EarlyExitBackbone(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden = self.encoder.config.hidden_size
        self.num_layers = self.encoder.config.num_hidden_layers
        self.classifiers = nn.ModuleList([nn.Linear(hidden, num_labels) for _ in range(self.num_layers)])
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hs = out.hidden_states[1:]
        return [clf(h[:,0,:]) for clf,h in zip(self.classifiers, hs)]

def train_backbone(train_texts, train_labels, num_labels, epochs=3, seed=42):
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = EarlyExitBackbone(MODEL_NAME, num_labels).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()
    enc = tok(list(train_texts), truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    labels = torch.tensor(list(train_labels)); n = len(train_labels)
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, 16):
            idx = perm[i:i+16]
            ii = enc['input_ids'][idx].to(DEVICE); am = enc['attention_mask'][idx].to(DEVICE); y = labels[idx].to(DEVICE)
            opt.zero_grad()
            logits_list = model(ii, am)
            loss = sum(loss_fn(lg, y) for lg in logits_list)
            loss.backward(); opt.step()
    return model, tok

def _entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p*torch.log(p+1e-12)).sum(dim=-1)

@torch.no_grad()
def run_deebert(model, tok, test_texts, test_labels, thresh=0.1):
    model.eval(); preds, exits, lat = [], [], []
    for t in test_texts:
        enc = tok(t, truncation=True, padding='max_length', max_length=128, return_tensors='pt').to(DEVICE)
        if DEVICE=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter()
        out = model.encoder(**enc, output_hidden_states=True)
        for l,h in enumerate(out.hidden_states[1:]):
            lg = model.classifiers[l](h[:,0,:])
            if _entropy(lg).item() < thresh or l==model.num_layers-1:
                preds.append(lg.argmax(-1).item()); exits.append(l+1); break
        if DEVICE=='cuda': torch.cuda.synchronize()
        lat.append((time.perf_counter()-t0)*1000)
    return _summarize('DeeBERT', preds, test_labels, exits, lat)

@torch.no_grad()
def run_pabee(model, tok, test_texts, test_labels, patience=3):
    model.eval(); preds, exits, lat = [], [], []
    for t in test_texts:
        enc = tok(t, truncation=True, padding='max_length', max_length=128, return_tensors='pt').to(DEVICE)
        if DEVICE=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter()
        out = model.encoder(**enc, output_hidden_states=True)
        last, cnt, fp, fl = None, 0, None, model.num_layers
        for l,h in enumerate(out.hidden_states[1:]):
            lg = model.classifiers[l](h[:,0,:]); p = lg.argmax(-1).item()
            cnt = cnt+1 if p==last else 1
            last = p
            if cnt>=patience or l==model.num_layers-1:
                fp, fl = p, l+1; break
        preds.append(fp); exits.append(fl)
        if DEVICE=='cuda': torch.cuda.synchronize()
        lat.append((time.perf_counter()-t0)*1000)
    return _summarize('PABEE', preds, test_labels, exits, lat)

@torch.no_grad()
def run_fastbert(model, tok, test_texts, test_labels, thresh=0.1):
    model.eval(); preds, exits, lat = [], [], []
    num_labels = model.classifiers[0].out_features
    max_ent = np.log(num_labels)
    for t in test_texts:
        enc = tok(t, truncation=True, padding='max_length', max_length=128, return_tensors='pt').to(DEVICE)
        if DEVICE=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter()
        out = model.encoder(**enc, output_hidden_states=True)
        for l,h in enumerate(out.hidden_states[1:]):
            lg = model.classifiers[l](h[:,0,:])
            ne = (_entropy(lg)/max_ent).item()
            if ne < thresh or l==model.num_layers-1:
                preds.append(lg.argmax(-1).item()); exits.append(l+1); break
        if DEVICE=='cuda': torch.cuda.synchronize()
        lat.append((time.perf_counter()-t0)*1000)
    return _summarize('FastBERT', preds, test_labels, exits, lat)

def _summarize(name, preds, labels, exits, lat):
    return {'model':name,'accuracy':float(accuracy_score(labels,preds)),
            'f1_macro':float(f1_score(labels,preds,average='macro')),
            'avg_exit_layer':float(np.mean(exits)),
            'latency_ms_mean':float(np.mean(lat)),'latency_ms_std':float(np.std(lat)),
            'raw_latencies':lat}

## Model 5: Dynamic NN yang Diusulkan (ACT) + Ablasi

In [ ]:
# Sel 7 -- Dynamic NN dengan Adaptive Computation Time + mode ablasi
class DynamicNN(nn.Module):
    def __init__(self, model_name, num_labels, use_act=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden = self.encoder.config.hidden_size
        self.num_layers = self.encoder.config.num_hidden_layers
        self.use_act = use_act
        self.halting_units = nn.ModuleList([nn.Linear(hidden,1) for _ in range(self.num_layers)])
        self.classifier = nn.Linear(hidden, num_labels)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hs = out.hidden_states[1:]
        if not self.use_act:
            fh = hs[-1][:,0,:]
            return self.classifier(fh), float(self.num_layers), torch.tensor(0.0, device=input_ids.device)
        bs = input_ids.size(0)
        cum = torch.zeros(bs, device=input_ids.device); rem = torch.ones(bs, device=input_ids.device)
        wh = torch.zeros(bs, hs[0].size(-1), device=input_ids.device)
        nup = torch.zeros(bs, device=input_ids.device)
        run = torch.ones(bs, dtype=torch.bool, device=input_ids.device)
        for l,h in enumerate(hs):
            ch = h[:,0,:]
            p = torch.sigmoid(self.halting_units[l](ch)).squeeze(-1)
            last = (l==self.num_layers-1)
            newc = cum + p*run.float()
            halt = (newc>=1.0-1e-3)|last
            w = torch.where(halt&run, rem, p*run.float())
            wh = wh + w.unsqueeze(-1)*ch
            nup = nup + run.float()
            rem = torch.where(run, rem-p, rem)
            cum = torch.where(run, newc, cum)
            run = run & (~halt)
            if not run.any(): break
        logits = self.classifier(wh)
        return logits, nup.mean().item(), nup.mean()

def train_dynamic_nn(train_texts, train_labels, num_labels, use_act=True, epochs=5, tau=0.01, seed=42):
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = DynamicNN(MODEL_NAME, num_labels, use_act).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
    ce = nn.CrossEntropyLoss()
    enc = tok(list(train_texts), truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    labels = torch.tensor(list(train_labels)); n = len(train_labels)
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, 16):
            idx = perm[i:i+16]
            ii = enc['input_ids'][idx].to(DEVICE); am = enc['attention_mask'][idx].to(DEVICE); y = labels[idx].to(DEVICE)
            opt.zero_grad()
            logits, _, ponder = model(ii, am)
            loss = ce(logits, y) + tau*ponder
            loss.backward(); opt.step()
    return model, tok

@torch.no_grad()
def evaluate_dynamic_nn(model, tok, test_texts, test_labels):
    model.eval(); preds, exits, lat = [], [], []
    for t in test_texts:
        enc = tok(t, truncation=True, padding='max_length', max_length=128, return_tensors='pt').to(DEVICE)
        if DEVICE=='cuda': torch.cuda.synchronize()
        t0=time.perf_counter()
        logits, avgl, _ = model(enc['input_ids'], enc['attention_mask'])
        if DEVICE=='cuda': torch.cuda.synchronize()
        lat.append((time.perf_counter()-t0)*1000)
        preds.append(logits.argmax(-1).item()); exits.append(avgl)
    name = 'DynamicNN-ACT' if model.use_act else 'DynamicNN-noACT(ablation)'
    return {'model':name,'accuracy':float(accuracy_score(test_labels,preds)),
            'f1_macro':float(f1_score(test_labels,preds,average='macro')),
            'avg_exit_layer':float(np.mean(exits)),
            'latency_ms_mean':float(np.mean(lat)),'latency_ms_std':float(np.std(lat)),
            'raw_latencies':lat}

## Jalankan Semua Eksperimen per Bahasa

In [ ]:
# Sel 8 -- Pipeline lengkap per bahasa (ganti LANG di sini: 'id', 'en', atau 'ms')
LANG = 'id'  # <-- ubah ke 'en' lalu 'ms' dan jalankan ulang sel ini untuk tiap bahasa

train_texts, train_labels_raw, test_texts, test_labels_raw = DATASETS[LANG]
le = LabelEncoder()
train_labels = le.fit_transform(train_labels_raw).tolist()
test_labels = le.transform(test_labels_raw).tolist()
num_labels = len(le.classes_)
print(f'Bahasa: {LANG} | Kelas: {le.classes_} | Train: {len(train_texts)} | Test: {len(test_texts)}')
assert num_labels == 2, 'ERROR: jumlah kelas bukan 2 -- cek ulang file upload!'

results = {}

print('\n[1/5] mBERT statis...')
results['mbert'] = fine_tune_mbert(train_texts, train_labels, test_texts, test_labels, num_labels)

print('\n[2/5] Melatih backbone early-exit bersama...')
ee_model, ee_tok = train_backbone(train_texts, train_labels, num_labels)
print('  Menjalankan DeeBERT...'); results['deebert'] = run_deebert(ee_model, ee_tok, test_texts, test_labels)
print('  Menjalankan PABEE...'); results['pabee'] = run_pabee(ee_model, ee_tok, test_texts, test_labels)
print('  Menjalankan FastBERT...'); results['fastbert'] = run_fastbert(ee_model, ee_tok, test_texts, test_labels)

print('\n[3/5] Dynamic NN (dengan ACT)...')
dnn, dnn_tok = train_dynamic_nn(train_texts, train_labels, num_labels, use_act=True)
results['dynamic_nn'] = evaluate_dynamic_nn(dnn, dnn_tok, test_texts, test_labels)

print('\n[4/5] Dynamic NN ABLASI (tanpa ACT)...')
dnn_noact, dnn_noact_tok = train_dynamic_nn(train_texts, train_labels, num_labels, use_act=False)
results['dynamic_nn_ablation'] = evaluate_dynamic_nn(dnn_noact, dnn_noact_tok, test_texts, test_labels)

print('\n[5/5] Uji signifikansi statistik (paired t-test, BUKAN McNemar)...')
a = np.array(results['mbert']['raw_latencies']); b = np.array(results['dynamic_nn']['raw_latencies'][:len(a)])
t_stat, p_val = stats.ttest_rel(a, b[:len(a)] if len(b)>=len(a) else np.pad(b,(0,len(a)-len(b)),mode='edge'))
results['significance_dnn_vs_mbert'] = {'paired_t_test': {'statistic': float(t_stat), 'p_value': float(p_val)}}

results['_metadata'] = {'language': LANG, 'schema': '2-class (positive/negative)', 'n_train': len(train_texts), 'n_test': len(test_texts), 'classes': list(le.classes_)}

summary = {k: {kk:vv for kk,vv in v.items() if kk!='raw_latencies'} if isinstance(v,dict) else v for k,v in results.items()}
with open(f'results_{LANG}.json','w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'\n✅ SELESAI untuk bahasa: {LANG}. Hasil tersimpan di results_{LANG}.json')
print(json.dumps(summary, indent=2, default=str))

In [ ]:
# Sel 9 -- Download hasil (jalankan setelah Sel 8 selesai untuk SETIAP bahasa: id, en, ms)
from google.colab import files
files.download(f'results_{LANG}.json')

## Petunjuk Terakhir

1. Jalankan Sel 8 dan Sel 9 untuk `LANG='id'`, lalu ubah ke `LANG='en'`, lalu `LANG='ms'` (ulangi Sel 8 dan 9 tiga kali total).
2. Anda akan punya 3 file: `results_id.json`, `results_en.json`, `results_ms.json`.
3. **Kirim ketiga file JSON ini apa adanya** (jangan diedit/diketik ulang manual) ke Claude untuk disusun jadi draf Jurnal 1.
4. Setiap file berisi `_metadata` yang mencantumkan skema kelas dan ukuran data -- ini membantu verifikasi bahwa hasil benar-benar berasal dari notebook ini.